# Molecule Enuermators

Application to enumerate given query molecules. 

The Custom Molecule Enumerator allows user to choose the site to enumerate with building blocks filtered based on given constraints. 

The Automated Molecule Enumerator, however, automates the substructure identification based on a reaction data and enumerates possible substructure compositions with building blocks filtered based on similarity to the substructures.

#### Helper Functions and imports

In [ ]:
from rdkit.Chem import MolFromSmiles, MolToSmiles, MolFromSmarts, Draw, AllChem, AddHs, RemoveHs, rdFMCS

import ipywidgets as widgets
import nglview as nv
import utils

scaffold = MolFromSmiles('O=C(COS(=O)(NC)=O)NC1=NNC(C2=CC=CC=C2OCC)=C1')
# Example inputs
penicillin = 'CC1(C(N2C(S1)C(C2=O)NC(=O)CC3=CC=CC=C3)C(=O)O)C'
substructure = 'c1ccccc1CC(=O)'

In [ ]:
## helper functions
# rgb color dict for rdkit
color_dict = {
    'blue': (0.19, 0.51, 0.70),
    'purple': (0.68, 0.45, 0.8),
    'pink': (0.94, 0.32, 0.65),
    'green': (0.48, 0.68, 0.35),
    'yellow': (0.81, 0.82, 0.0),
    'red': (0.95, 0.42, 0.19),
    'orange': (0.93, 0.69, 0.17)
}

# rule based constraints
rules = {
    'MW': (0, 500), # molecular weight
    'HBD': (0, 5), # hydrogen bond donors
    'HBA': (0, 10), # hydrogen bond acceptors
    'TPSA': (0, 200), # topological polar surface area
    'RotB': (0, 10), # rotatable bonds
    'Rings': (0, 10), # number of rings
    'ArRings': (0, 5), # number of aromatic rings
    'Chiral': (0, 5), # number of chiral centers
}

# reaction based constraints
reactions = utils.load_reactions_from_json('reactions/reactions.json')
reactions = [r for r in reactions if r.is_valid()]
reaction_tags = [r.tags for r in reactions if r.is_valid()]
reaction_tags = list(set([tag for tags in reaction_tags for tag in tags]))

def get_nglview_mol(smi):
    mol = MolFromSmiles(smi)
    mol = AddHs(mol)
    AllChem.EmbedMolecule(mol)
    mol = RemoveHs(mol)
    return nv.show_rdkit(mol)

def text2svg(*args, fill: str = "#fff", font: str = "sans-serif", size: float = 16,
             ratio: float = 1, text_anchor: str = "middle",
             background_fill: str = "firebrick", background_opacity: float = 1.0,
             width: float = 350, height: float = 100, 
             ):
    """
        Convert text to svg+xml format.
    """
    svg_out = f'<svg version="1.1" width="{width}" height="{height}" xmlns="http://www.w3.org/2000/svg">'
    svg_out += f'<rect width="80%" height="100%" fill="{background_fill}" opacity="{background_opacity}" x="10%" rx="20" ry="20" />'
    for i, text in enumerate(args):
        svg_out += f'<text x="{width//2}" y="{(height//2) + (i * size * ratio)}" font-family="{font}" font-size="{size}" text-anchor="{text_anchor}" fill="{fill}">{text}</text>'
    svg_out += '</svg>'
    return svg_out.encode('utf-8')

def get_svg_mol(mol, sub_mol=None, sub_mol_color='green', legend='', show_idx=False):
    '''
        Get svg image of a molecule with a substructure highlighted.
    '''
    sub_mol_color = color_dict[sub_mol_color]
    if isinstance(mol, str):
       mol = MolFromSmiles(mol)
    AllChem.Compute2DCoords(mol)
    if sub_mol is not None:
        if isinstance(sub_mol, str):
            sub_struct = MolFromSmiles(sub_mol)
        else:
            sub_struct = sub_mol
        assert sub_struct is not None, 'Invalid substructure'
        assert mol.HasSubstructMatch(sub_struct), 'Substructure not found'
        hit_atoms = list(mol.GetSubstructMatch(sub_struct))
        hit_bonds = []
        for bond in sub_struct.GetBonds():
            a1 = hit_atoms[bond.GetBeginAtomIdx()]
            a2 = hit_atoms[bond.GetEndAtomIdx()]
            hit_bonds.append(mol.GetBondBetweenAtoms(a1, a2).GetIdx())
    else:
        hit_atoms = []
        hit_bonds = []
    if show_idx:
        for i, atom in enumerate(mol.GetAtoms()):
            atom.SetProp('atomNote', str(i))
    drawing = Draw.MolDraw2DSVG(350, 100)
    drawing.DrawMolecule(mol, highlightAtoms=hit_atoms, highlightBonds=hit_bonds,
                          highlightAtomColors={i: sub_mol_color for i in hit_atoms},
                          highlightBondColors={i: sub_mol_color for i in hit_bonds},
                          legend=legend)
    drawing.FinishDrawing()
    svg = drawing.GetDrawingText()
    return svg.encode('utf-8')

def get_svg_mol_with_bbs(mol, bb1, bb2, bb_colors=['red', 'green'], legend=''):
    '''
        Similar to get_svg_mol, but instead of highlighting a substructure,
        it highlights the building blocks.
    '''
    bb_colors = [color_dict[c] for c in bb_colors]
    if isinstance(mol, str):
       mol = MolFromSmiles(mol)
    AllChem.Compute2DCoords(mol)
    if isinstance(bb1, str):
        bb1 = MolFromSmiles(bb1)
    if isinstance(bb2, str):
        bb2 = MolFromSmiles(bb2)
    assert bb1 is not None, 'Invalid building block 1'
    assert bb2 is not None, 'Invalid building block 2'
    
    mcs1 = rdFMCS.FindMCS([mol, bb1])
    mcs2 = rdFMCS.FindMCS([mol, bb2])
    smarts1 = mcs1.smartsString
    smarts2 = mcs2.smartsString
    bb1 = MolFromSmarts(smarts1)
    bb2 = MolFromSmarts(smarts2)
    hit_atoms1 = list(mol.GetSubstructMatch(bb1))
    hit_atoms2 = list(mol.GetSubstructMatch(bb2))
    hit_bonds1 = []
    hit_bonds2 = []
    for bond in bb1.GetBonds():
        a1 = hit_atoms1[bond.GetBeginAtomIdx()]
        a2 = hit_atoms1[bond.GetEndAtomIdx()]
        try:
            hit_bonds1.append(mol.GetBondBetweenAtoms(a1, a2).GetIdx())
        except:
            pass
    for bond in bb2.GetBonds():
        a1 = hit_atoms2[bond.GetBeginAtomIdx()]
        a2 = hit_atoms2[bond.GetEndAtomIdx()]
        try:
            hit_bonds2.append(mol.GetBondBetweenAtoms(a1, a2).GetIdx())
        except:
            pass
    drawing = Draw.MolDraw2DSVG(350, 150)
    drawing.DrawMolecule(mol, highlightAtoms=hit_atoms1+hit_atoms2, highlightBonds=hit_bonds1+hit_bonds2,
                          highlightAtomColors={**{i: bb_colors[0] for i in hit_atoms1}, **{i: bb_colors[1] for i in hit_atoms2}},
                          highlightBondColors={**{i: bb_colors[0] for i in hit_bonds1}, **{i: bb_colors[1] for i in hit_bonds2}},
                          legend=legend)
    drawing.FinishDrawing()
    svg = drawing.GetDrawingText()
    return svg.encode('utf-8')

def image_slider(mols, i):
    return get_svg_mol(MolToSmiles(mols[i]))


## IPython Widgets

### Custom Molecule Enumerator


In [ ]:
## Custom Enumerator application using ipywidgets
# header
header = widgets.HTML(value="<h1>Custom Enumerator</h1>", layout=widgets.Layout(margin='0px 0px 0px 400px'))

# input widgets
molecule_input = widgets.Textarea(value='CC1(C(N2C(S1)C(C2=O)NC(=O)CC3=CC=CC=C3)C(=O)O)C', 
                                  placeholder='Type a molecule SMILES here', 
                                  description='Molecule:', layout=widgets.Layout(width='auto'))
building_block_source = widgets.Dropdown(options=['test', 'US_stocks', 'EU_stocks', 'global_stocks'], 
                                         layout=widgets.Layout(width='auto'), 
                                         description='BB source:')
reaction_sites = widgets.Textarea(value='19, 20, 21', 
                                  placeholder='Atom indices for reaction sites',
                                  description='Rxn Sites:', layout=widgets.Layout(width='auto'))

# reaction tags
rxn_tag_desc = widgets.HTML(value='Reaction Tags:',
                            layout=widgets.Layout(margin='0px 0px 0px 0px', width='150px'))
rxn_tags = widgets.TagsInput(value=['amide coupling', 'amide', 'C-N bond formation', 'C-N', 
                                    'alkylation', 'N-arylation', 'azole', 'amination'],
                             allowed_tags=reaction_tags,
                             allow_duplicates=False,
                             layout=widgets.Layout(width='auto'),)
rxn_inputs = widgets.HBox([rxn_tag_desc, rxn_tags], 
                         layout=widgets.Layout(margin='0px 0px 0px 0px', height='100px'))

# save file name
save_file_name = widgets.Text(value='enumerated_molecules.csv', 
                              placeholder='Enter file name to save enumerated molecules', 
                              description='Save as:', 
                              layout=widgets.Layout(width='auto', margin='20px 0px 0px 0px'))

# substruct rule
substructure_rules = widgets.Textarea(value='c1ccccc1, CC', 
                                      placeholder='Substructure constraint for enumeration', 
                                      description='Struct. Rule:', layout=widgets.Layout(width='auto'))

# rules as sliders
rule_sliders = {}
for rule, (min_val, max_val) in rules.items():
    if rule in ['MW', 'TPSA']:
        rule_sliders[rule] = widgets.FloatRangeSlider(value=[min_val, max_val], 
                                                      min=min_val, 
                                                      max=max_val, 
                                                      step=1, 
                                                      description=rule,
                                                      readout_format='.1f')
    else:
        rule_sliders[rule] = widgets.IntRangeSlider(value=[min_val, max_val], 
                                                    min=min_val, 
                                                    max=max_val, 
                                                    step=1, 
                                                    description=rule)
    rule_sliders[rule].layout.width = 'auto'
    rule_sliders[rule].layout.margin = '10px 0px 0px 0px'
    rule_sliders[rule].style.description_width = 'auto'
    rule_sliders[rule].style.handle_color = 'lightblue'

# buttons
update_button = widgets.Button(description='Update', 
                               layout=widgets.Layout(margin='0px 0px 0px 40px'),
                               style=widgets.ButtonStyle(button_color='lightblue'))
enumeration_button = widgets.Button(description='Enumerate',
                                    layout=widgets.Layout(margin='0px 0px 0px 40px'),
                                    style=widgets.ButtonStyle(button_color='orange'))
save_button = widgets.Button(description='Save',
                             layout=widgets.Layout(margin='0px 0px 0px 40px'),
                             style=widgets.ButtonStyle(button_color='lightgreen'))
button_box = widgets.HBox([update_button, enumeration_button, save_button], 
                          layout=widgets.Layout(margin='10px 0px 0px 50px', height='50px', width='500px'))

# molecule box
molecule_title = widgets.HTML(value='<h2>Input Molecule</h2>', 
                              layout=widgets.Layout(display='flex', margin='0px 0px 0px 210px'))
molecule_image = widgets.Image(value=text2svg('Click Update Button!', fill='black', background_fill='white'), 
                               layout=widgets.Layout(display='flex', margin='-20px 0px 0px 0px'),
                               format='svg+xml',)
molecule_box = widgets.VBox([molecule_title, molecule_image],)

# enumeration box with slider to view resulting enumeration after clicking the button
enumeration_title = widgets.HTML(value='<h2>Enumeration Results</h2>', 
                                 layout=widgets.Layout(margin='0px 0px 0px 160px'))
enumeration_slider = widgets.IntSlider(layout=widgets.Layout(margin='0px 0px 0px 150px'),)
enumeration_image = widgets.Image(value=text2svg('Click Enumerate Button!', fill='black', background_fill='white'),
                                  layout=widgets.Layout(display='flex', margin='0px 0px 0px 0px'),
                                  format='svg+xml',)
enumeration_box = widgets.VBox([enumeration_title, enumeration_slider, enumeration_image], )

# events
def click_on_update(b):
    '''
        Update button event. It will update the molecule image after changing the 
        molecule or substructure.
    '''
    molecule_image.value = get_svg_mol(molecule_input.value, show_idx=True)
    enumeration_image.value = text2svg('Click Enumerate Button!', fill='black', background_fill='white')
update_button.on_click(click_on_update)

def click_on_enumerate(b):
    '''
        Enumerate button event. It will call the enumerator function for 
        the given inputs.
    '''
    molecule = molecule_input.value
    building_blocks = building_block_source.value
    sites = [int(i) for i in reaction_sites.value.replace(' ', '').split(',')] if reaction_sites.value else []
    struct_rules = [s for s in substructure_rules.value.replace(' ', '').split(',')] if substructure_rules.value else []
    rules = {rule: rule_sliders[rule].value for rule in rule_sliders}
    reaction_tags = rxn_tags.value
    enumerator = utils.custom_enumerator(molecule, building_blocks, sites, reaction_tags, rules, struct_rules)
    if not enumerator.enumerated_molecules:
        enumeration_slider.max = 0
        enumeration_slider.value = 0
        enumeration_image.value = text2svg('No Enumerations!', 'Check Rules and Substructure!')
        del enumerator
        def click_on_save(b):
            enumeration_image.value = text2svg('No Enumerations!', "Couldn't Save!", background_fill='red')
        save_button.on_click(click_on_save)
    else:
        enumeration_slider.max = len(enumerator.enumerated_molecules) - 1
        enumeration_slider.step = 1
        enumeration_image.value = text2svg("Slide to view enumerated molecules!", background_fill='green')

        def on_value_change(change):
            enumeration_slider.value = change['new']
            enumeration_image.value = get_svg_mol_with_bbs(enumerator.enumerated_molecules[change['new']][0],
                                                           enumerator.molecule,
                                                           enumerator.enumerated_molecules[change['new']][1],
                                                           bb_colors=['red', 'green'],
                                                           legend=enumerator.enumerated_molecules[change['new']][-2])
        enumeration_slider.observe(on_value_change, names='value')

    def click_on_save(b):
        '''
            Save button event. It will save the enumerated molecules to a file.
        '''
        enumerator.save_results(save_file_name.value)
        enumeration_image.value = text2svg(f'Saved to {save_file_name.value}!', background_fill='green')
    save_button.on_click(click_on_save)
enumeration_button.on_click(click_on_enumerate)


# grid layout
grid = widgets.GridspecLayout(12, 4, height='700px', width='1100px', grid_gap='5px')
grid[0, :] = header
grid[1, :2] = molecule_input
grid[2, :2] = building_block_source
grid[3, :2] = reaction_sites
grid[4, :2] = substructure_rules
grid[5, :2] = rxn_inputs
grid[10, :2] = save_file_name
for i, (rule, slider) in enumerate(rule_sliders.items()):
    grid[(i%4)+6, i//4] = slider
grid[11, :2] = button_box
grid[1:5, 2:] = molecule_box
grid[5:, 2:] = enumeration_box

grid


### Automated Molecule Enumerator


In [ ]:
## Automated Enumerator application using ipywidgets
# header
header = widgets.HTML(value="<h1>Automated Enumerator</h1>", layout=widgets.Layout(margin='0px 0px 0px 400px'))

# input widgets
molecule_input = widgets.Textarea(value='CC1(C(N2C(S1)C(C2=O)NC(=O)CC3=CC=CC=C3)C(=O)O)C', 
                                  placeholder='Type a molecule SMILES here', 
                                  description='Molecule:', layout=widgets.Layout(width='auto'))
building_block_source = widgets.Dropdown(options=['test', 'US_stocks', 'EU_stocks', 'global_stocks'], 
                                         layout=widgets.Layout(width='auto'), 
                                         description='BB source:')
custom_comp_sites = widgets.Textarea(value='',
                                     placeholder='Custom sites to split the molecule, e.g. 9,10; 6,9',
                                     description='Custom Sites:', layout=widgets.Layout(width='auto'))
similarity_threshold = widgets.FloatSlider(value=0.15, 
                                           min=0.0, 
                                           max=1.0, 
                                           step=0.01, 
                                           description='Sim Cutoff:', 
                                           readout_format='.2f',
                                           style=widgets.SliderStyle(handle_color='lightblue'),
                                           layout=widgets.Layout(width='auto'))
rxn_tag_desc = widgets.HTML(value='Reaction Tags:',
                            layout=widgets.Layout(margin='0px 0px 0px 0px', width='150px'))
rxn_tags = widgets.TagsInput(value=['amide coupling', 'amide', 'C-N bond formation', 'C-N', 
                                    'alkylation', 'N-arylation', 'azole', 'amination'],
                             allowed_tags=reaction_tags,
                             allow_duplicates=False,
                             layout=widgets.Layout(width='auto'),)
rxn_inputs = widgets.HBox([rxn_tag_desc, rxn_tags], 
                         layout=widgets.Layout(margin='0px 0px 0px 0px', height='100px'))

# save file name
save_file_name = widgets.Text(value='enumerated_molecules.csv', 
                              placeholder='Enter file name to save enumerated molecules', 
                              description='Save as:', layout=widgets.Layout(width='auto'))

# buttons
update_button = widgets.Button(description='Update', 
                               layout=widgets.Layout(margin='0px 0px 0px 40px'),
                               style=widgets.ButtonStyle(button_color='lightblue'))
enumeration_button = widgets.Button(description='Enumerate',
                                    layout=widgets.Layout(margin='0px 0px 0px 40px'),
                                    style=widgets.ButtonStyle(button_color='orange'))
save_button = widgets.Button(description='Save',
                             layout=widgets.Layout(margin='0px 0px 0px 40px'),
                             style=widgets.ButtonStyle(button_color='lightgreen'))
button_box = widgets.HBox([update_button, enumeration_button, save_button], 
                          layout=widgets.Layout(margin='10px 0px 0px 50px', height='50px', width='500px'))

# molecule box
molecule_title = widgets.HTML(value='<h2>Molecule</h2>', 
                              layout=widgets.Layout(display='flex', margin='0px 0px 0px 210px'))
molecule_image = widgets.Image(value=text2svg('Click Update Button!', fill='black', background_fill='white'),
                               layout=widgets.Layout(display='flex', margin='-20px 0px 0px 0px'),
                               format='svg+xml',)
molecule_box = widgets.VBox([molecule_title, molecule_image],)

# enumeration box with slider to view resulting enumeration after clicking the button
enumeration_title = widgets.HTML(value='<h2>Enumeration Results</h2>', 
                                 layout=widgets.Layout(margin='0px 0px 0px 160px'))
enumeration_slider = widgets.IntSlider(layout=widgets.Layout(margin='0px 0px 0px 150px'),)
enumeration_image = widgets.Image(value=text2svg('Click Enumerate Button!', fill='black', background_fill='white'),
                                  layout=widgets.Layout(display='flex', margin='0px 0px 0px 0px'),
                                  format='svg+xml',)
enumeration_box = widgets.VBox([enumeration_title, enumeration_slider, enumeration_image], 
                               layout=widgets.Layout(margin='0px 0px 0px 0px'))

# events
def click_on_update(b):
    '''
        Update button event. It will update the molecule image after changing the 
        molecule or substructure.
    '''
    molecule_image.value = get_svg_mol(molecule_input.value, show_idx=True)
    enumeration_image.value = text2svg('Click Enumerate Button!', fill='black', background_fill='white')
update_button.on_click(click_on_update)

def click_on_enumerate(b):
    '''
            Enumerate button event. It will call the enumerator function for 
            the given inputs.
    '''
    molecule = molecule_input.value
    building_blocks = building_block_source.value
    sim_threshold = similarity_threshold.value
    # custom_sites = [tuple(map(int, site.replace(' ', '').split(','))) 
    #                 for site in custom_comp_sites.value.split('\n') if site] if custom_comp_sites.value else []
    custom_sites = [tuple(map(int, site.replace(' ', '').split(',')))
                    for site in custom_comp_sites.value.split(';') if site] if custom_comp_sites.value else []
    reaction_tags = rxn_tags.value
    enumerator = automated_enumerator(molecule, building_blocks, reaction_tags, custom_sites, 10, sim_threshold)
    if not enumerator.enumerated_molecules:
        enumeration_slider.max = 0
        enumeration_slider.value = 0
        enumeration_image.value = text2svg('No Enumerations!', 'Check Rules and Substructure!')
        del enumerator
        def click_on_save(b):
                    enumeration_image.value = text2svg('No Enumerations!', "Couldn't Save!", background_fill='red')
        save_button.on_click(click_on_save)
    else:
        enumeration_slider.max = len(enumerator.enumerated_molecules) - 1
        enumeration_slider.step = 1
        enumeration_image.value = text2svg("Slide to view enumerated molecules!", background_fill='green')

        def on_value_change(change):
            enumeration_slider.value = change['new']
            enumeration_image.value = get_svg_mol_with_bbs(*enumerator.enumerated_molecules[change['new']][:3],
                                                            bb_colors=['purple', 'green'],
                                                            legend=enumerator.enumerated_molecules[change['new']][3])
        enumeration_slider.observe(on_value_change, names='value')

    def click_on_save(b):
        '''
            Save button event. It will save the enumerated molecules to a file.
        '''
        enumerator.save_results(save_file_name.value)
        enumeration_image.value = text2svg(f'Saved to {save_file_name.value}!', background_fill='green')
    save_button.on_click(click_on_save)
enumeration_button.on_click(click_on_enumerate)

# grid layout
grid = widgets.GridspecLayout(12, 4, height='600px', width='1100px', grid_gap='5px')
grid[0, :] = header
grid[1, :2] = molecule_input
grid[2, :2] = building_block_source
grid[3, :2] = custom_comp_sites
grid[4, :2] = similarity_threshold
grid[5, :2] = rxn_inputs
grid[9, :2] = save_file_name
grid[10, :2] = button_box
grid[1:5, 2:] = molecule_box
grid[5:, 2:] = enumeration_box

grid


## Dash Apps

### Automated Enumerator

In [ ]:
# Automated Enumerator using Dash
from dash import dcc, html, Dash, Input, Output, State, callback_context, no_update

import dash_bootstrap_components as dbc
import utils

# List of unique reaction tags
reactions = utils.load_reactions_from_json('reactions/reactions.json')
reactions = [r for r in reactions if r.is_valid()]
reaction_tags = [r.tags for r in reactions if r.is_valid()]
reaction_tags = list(set([tag for tags in reaction_tags for tag in tags]))


app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = dbc.Container(
    [
        html.H1("Automated Enumerator", style={'text-align': 'center', 'margin': '10px 0px 10px 0px'}),
        dbc.Row(
            [
                dbc.Col(
                    [
                        dbc.Row(
                            [
                                dbc.Col(html.Label("Molecule"), width=3),
                                dbc.Col(dbc.Input(type="text", id="molecule-input", placeholder="Enter molecule", 
                                                  value="CC1(C(N2C(S1)C(C2=O)NC(=O)CC3=CC=CC=C3)C(=O)O)C",
                                                  style={'fontSize': '14px'}),
                                        width=9)
                            ],
                            className="mb-3"
                        ),
                        dbc.Row(
                            [
                                dbc.Col(html.Label("BB source"), width=3),
                                dbc.Col(
                                    dbc.Select(
                                        id="bb-source-select",
                                        options=[
                                            {"label": "test", "value": "test"},
                                            {"label": "US_stocks", "value": "US_stocks"},
                                            {"label": "EU_stocks", "value": "EU_stocks"},
                                            {"label": "global_stocks", "value": "global_stocks"}
                                        ],
                                        value="test"
                                    ),
                                    width=9
                                )
                            ],
                            className="mb-3"
                        ),
                        dbc.Row(
                            [
                                dbc.Col(html.Label("Custom Sites"), width=3),
                                dbc.Col(dbc.Input(type="text", id="custom-sites-input", 
                                                  placeholder="Custom sites to split the molecule, e.g., 9,10; 6,9",
                                                  style={'fontSize': '14px'}), 
                                        width=9)
                            ],
                            className="mb-3"
                        ),
                        dbc.Row(
                            [
                                dbc.Col(html.Label("Sim Cutoff"), width=3),
                                dbc.Col(
                                    dcc.Slider(
                                        id='sim-cutoff-slider',
                                        min=0,
                                        max=1,
                                        step=0.01,
                                        value=0.15,
                                        marks={i / 10: str(i / 10) for i in range(11)},
                                        tooltip={'placement': 'bottom', 'always_visible': False}
                                    ),
                                    width=9
                                )
                            ],
                            className="mb-3"
                        ),
                        dbc.Row(
                            [
                                dbc.Col(html.Label("Reaction Tags"), width=3),
                                dbc.Col(
                                    dcc.Dropdown(
                                        id='reaction-tags-dropdown',
                                        options=[{"label": tag, "value": tag} for tag in reaction_tags],
                                        value=["amide coupling", "amide", "C-N bond formation", "C-N", 
                                               "alkylation", "N-arylation", "azole", "amination"],
                                        multi=True,
                                    ),
                                    width=9
                                )
                            ],
                            className="mb-3"
                        ),
                        dbc.Row(
                            [
                                dbc.Col(html.Label("Save as"), width=3),
                                dbc.Col(dbc.Input(type="text", id="save-as-input", placeholder="Filename", 
                                                  value="enumerated_molecules.csv", style={'fontSize': '14px'}),
                                        width=9)
                            ],
                            className="mb-3"
                        ),
                        dbc.Row(
                            [
                                dbc.Col(dbc.Button("Update", id="update-button", color="primary", className="mr-2",
                                                   style={'width': '100%', 'text-align': 'center'}), width=3),
                                dbc.Col(dbc.Button("Enumerate", id="enumerate-button", color="warning", className="mr-2",
                                                   style={'width': '100%', 'text-align': 'center'}), width=3),
                                dbc.Col(dbc.Button("Save", id="save-button", color="success",
                                                   style={'width': '100%', 'text-align': 'center'}), width=3)
                            ],
                            className="mb-3",
                            justify='end',
                        ),
                    ],
                    md=6,
                    style={'height': '100%', 'justify_content': 'center', 'margin': '20px 0px 0px 0px'}
                ),
                dbc.Col(
                    [
                        html.H3("Molecule", style={'text-align': 'center'}),
                        html.Div(id='molecule-svg', style={'height': '150px', 'text-align': 'center'}),
                        html.H3("Enumeration Results", style={'text-align': 'center', 'margin': '20px 0px 20px 0px'}),
                        html.Div(dcc.Slider(id='enumeration-results-slider', min=0, max=0, step=1, value=0,
                                            tooltip={'placement': 'bottom', 'always_visible': False}), 
                                 style={'width': '70%', 'justify-content': 'center', 'margin': '0 auto'}),
                        html.Div(id='enumeration-svg', style={'height': '150px', 'text-align': 'center'}),
                        dcc.Download(id="download-enumerations"),
                        dcc.Store(id='enumeration-store')
                    ],
                    md=6,
                ),
            ],
            style={'height': '100%', 'justify_content': 'center', 'margin': '40px 0px 0px 0px'}
        ),
    ],
    fluid=True,
    style={'width': '80%'}
)

@app.callback(
    [Output('molecule-svg', 'children'),
     Output('molecule-svg', 'style'),
     Output('enumeration-svg', 'children'),
     Output('enumeration-svg', 'style'),
     Output('enumeration-results-slider', 'max'),
     Output('enumeration-results-slider', 'marks'),
     Output('enumeration-store', 'data'),
     Output('download-enumerations', 'data')],
    [Input('update-button', 'n_clicks'),
     Input('enumerate-button', 'n_clicks'),
     Input('enumeration-results-slider', 'value'),
     Input('save-button', 'n_clicks')],
    [State('molecule-input', 'value'),
     State('bb-source-select', 'value'),
     State('custom-sites-input', 'value'),
     State('sim-cutoff-slider', 'value'),
     State('reaction-tags-dropdown', 'value'),
     State('save-as-input', 'value'),
     State('enumeration-store', 'data')]
)
def update_output(update_clicks, enumerate_clicks, enumeration_value, save_clicks, 
                  molecule, building_blocks, custom_sites, sim_threshold, reaction_tags, save_as, enumerated_molecules):
    ctx = callback_context
    if not ctx.triggered:
        mol_img = 'Click Update to View the Molecule'
        mol_style = {'width': '75%', 'height': '175px', 'text-align': 'center', 
                     'fontSize': '20px', 'fontFamily': 'sans-serif', 'alignItems': 'center', 
                     'justifyContent': 'center', 'display': 'flex', 'margin': '0 auto'}
        enum_img = 'Click Enumerate to View the Enumerations'
        enum_style = {'width': '75%', 'height': '200px', 'text-align': 'center', 
                     'fontSize': '20px', 'fontFamily': 'sans-serif', 'alignItems': 'center', 
                     'justifyContent': 'center', 'display': 'flex', 'margin': '0 auto'}
        return mol_img, mol_style, enum_img, enum_style, 0, {}, None, None
    else:
        button_id = ctx.triggered[0]['prop_id'].split('.')[0]
        if button_id == 'update-button':
            mol_img = html.Img(src=utils.get_svg_mol(molecule, show_idx=True),
                               style={'height': '100%', 'width': '80%', 'margin': '0 auto'})
            mol_style = {'height': '175px', 'text-align': 'center'}
            return mol_img, mol_style, no_update, no_update, no_update, no_update, no_update, no_update
        elif button_id == 'enumerate-button':
            custom_sites = [tuple(map(int, site.replace(' ', '').split(',')))
                            for site in custom_sites.split(';') if site] if custom_sites else []
            enumerator = utils.automated_enumerator(molecule, building_blocks, reaction_tags, custom_sites, 10, sim_threshold)
            if not enumerator.enumerated_molecules:
                mol_img = html.Img(src=utils.get_svg_mol(molecule, show_idx=True),
                                   style={'height': '100%', 'width': '80%', 'margin': '0 auto'})
                mol_style = {'height': '175px', 'text-align': 'center'}
                slider_max = 0
                slider_marks = {0: '0'}
                enum_img = "No Enumerations! Check Inputs!"
                enum_style = {'backgroundColor': 'firebrick', 'width': '75%', 'height': '150px', 
                              'text-align': 'center', 'fontSize': '24px', 'fontFamily': 'sans-serif', 
                              'borderRadius': '20px', 'alignItems': 'center', 'justifyContent': 'center', 
                              'display': 'flex', 'margin': '0 auto'}
                return mol_img, mol_style, enum_img, enum_style, slider_max, slider_marks, [], no_update
            else:
                mol_img = html.Img(src=utils.get_svg_mol(molecule, show_idx=True),
                                   style={'height': '100%', 'width': '75%', 'margin': '0 auto'})
                mol_style = {'height': '175px', 'text-align': 'center'}
                enum_img = "Slide to view enumerated molecules!"
                enum_style = {'backgroundColor': 'forestgreen', 'width': '80%', 'height': '150px', 
                              'text-align': 'center', 'fontSize': '24px', 'fontFamily': 'sans-serif', 
                              'borderRadius': '20px', 'alignItems': 'center', 'justifyContent': 'center', 
                              'display': 'flex', 'margin': '0 auto'}
                slider_max = len(enumerator.enumerated_molecules) - 1
                n = len(enumerator.enumerated_molecules)
                slider_marks = {i: str(i) for i in range(0, n, max(10, round(n//10, -1)))}
                enum_mols = enumerator.enumerated_molecules
                return mol_img, mol_style, enum_img, enum_style, slider_max, slider_marks, enum_mols, no_update
        elif button_id == 'enumeration-results-slider':
            if enumerated_molecules:
                img = utils.get_svg_mol_with_bbs(*enumerated_molecules[enumeration_value][:3],
                                                 bb_colors=['purple', 'green'],
                                                 legend=enumerated_molecules[enumeration_value][3])
                enum_img = html.Img(src=img,
                                    style={'height': '100%', 'width': '80%', 'margin': '0 auto'})
                enum_style = {'height': '200px', 'text-align': 'center'}
            else:
                enum_img = "No Enumerations! Couldn't Save!"
                enum_style = {'backgroundColor': 'firebrick', 'width': '75%', 'height': '150px', 
                              'text-align': 'center', 'fontSize': '24px', 'fontFamily': 'sans-serif', 
                              'borderRadius': '20px', 'alignItems': 'center', 'justifyContent': 'center', 
                              'display': 'flex', 'margin': '0 auto'}
            return no_update, no_update, enum_img, enum_style, no_update, no_update, no_update, no_update
        elif button_id == 'save-button':
            enum_img = f'Saved to {save_as}!'
            enum_style = {'backgroundColor': 'forestgreen', 'width': '75%', 'height': '150px',
                          'text-align': 'center', 'fontSize': '24px', 'fontFamily': 'sans-serif',
                          'borderRadius': '20px', 'alignItems': 'center', 'justifyContent': 'center', 
                          'display': 'flex', 'margin': '0 auto'}
            if enumerated_molecules:
                save_out = dcc.send_data_frame(utils.automated_enumerator_download_df(enumerated_molecules).to_csv, filename=save_as)
            else:
                save_out = None
            return no_update, no_update, enum_img, enum_style, no_update, no_update, no_update, save_out

if __name__ == '__main__':
    app.run(debug=False, jupyter_mode='tab', port=8050)


### Custom Enumerator

In [ ]:
# Custom Enumerator using Dash
from dash import dcc, html, Dash, Input, Output, State, callback_context, no_update

import dash_bootstrap_components as dbc
import utils

# List of unique reaction tags
reactions = utils.load_reactions_from_json('reactions/reactions.json')
reactions = [r for r in reactions if r.is_valid()]
reaction_tags = [r.tags for r in reactions if r.is_valid()]
reaction_tags = list(set([tag for tags in reaction_tags for tag in tags]))


app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = dbc.Container(
    [
        html.H1("Custom Molecule Enumerator", style={'text-align': 'center', 'margin': '10px 0px 10px 0px'}),
        dbc.Row(
            [
                dbc.Col(
                    [
                        dbc.Row(
                            [
                                dbc.Col(html.Label("Molecule"), width=3),
                                dbc.Col(dbc.Input(type="text", id="molecule-input", placeholder="Enter molecule", 
                                                  value="CC1(C(N2C(S1)C(C2=O)NC(=O)CC3=CC=CC=C3)C(=O)O)C",
                                                  style={'fontSize': '14px'}),
                                        width=9)
                            ],
                            className="mb-3"
                        ),
                        dbc.Row(
                            [
                                dbc.Col(html.Label("BB source"), width=3),
                                dbc.Col(
                                    dbc.Select(
                                        id="bb-source-select",
                                        options=[
                                            {"label": "test", "value": "test"},
                                            {"label": "US_stocks", "value": "US_stocks"},
                                            {"label": "EU_stocks", "value": "EU_stocks"},
                                            {"label": "global_stocks", "value": "global_stocks"}
                                        ],
                                        value="test"
                                    ),
                                    width=9
                                )
                            ],
                            style={'margin': '0px 0px 10px 0px'}
                        ),
                        dbc.Row(
                            [
                                dbc.Col(html.Label("Reaction Sites"), width=3),
                                dbc.Col(dbc.Input(type="text", id="reaction-sites-input", 
                                                  placeholder="Reacting atoms, e.g., 19, 20, 21",
                                                  style={'fontSize': '14px'}), 
                                        width=9)
                            ],
                            className="mb-3"
                        ),
                        dbc.Row(
                            [
                                dbc.Col(html.Label("Reaction Tags"), width=3),
                                dbc.Col(
                                    dcc.Dropdown(
                                        id='reaction-tags-dropdown',
                                        options=[{"label": tag, "value": tag} for tag in reaction_tags],
                                        value=["amide coupling", "amide", "C-N bond formation", "C-N", 
                                               "alkylation", "N-arylation", "azole", "amination"],
                                        multi=True,
                                    ),
                                    width=9
                                )
                            ],
                            className="mb-3"
                        ),
                        dbc.Row(
                            [
                                dbc.Col(html.Label('Struct. Rules'), width=3),
                                dbc.Col(dbc.Textarea(id="struct-rules-input", 
                                                     placeholder="Substructure constraints for BBs, e.g., c1ccccc1, CC",
                                                     value="", style={'fontSize': '14px'}),
                                        width=9)
                            ]
                        ),
                        dbc.Row(
                            [
                                dbc.Col(html.Label('MW'), width=1),
                                dbc.Col(dcc.RangeSlider(id='MW-slider', min=0, max=1000, step=0.1, value=[0, 1000],
                                                        marks={0: '0', 1000: '1000'},
                                                        tooltip={'placement': 'bottom', 'always_visible': True}),
                                        width=5),
                                dbc.Col(dcc.RangeSlider(id='TPSA-slider', min=0, max=200, step=0.1, value=[0, 200],
                                                        marks={0: '0', 200: '200'},
                                                        tooltip={'placement': 'bottom', 'always_visible': True}),
                                        width=5),
                                dbc.Col(html.Label('TPSA'), width=1),
                            ],
                            className="mb-3"
                        ),
                        dbc.Row(
                            [
                                dbc.Col(html.Label("HBD"), width=1),
                                dbc.Col(dcc.RangeSlider(id='HBD-slider', min=0, max=10, step=1, value=[0, 10],
                                                        marks={0: '0', 10: '10'},
                                                        tooltip={'placement': 'bottom', 'always_visible': True}),
                                        width=5),
                                dbc.Col(dcc.RangeSlider(id='HBA-slider', min=0, max=10, step=1, value=[0, 10],
                                                        marks={0: '0', 10: '10'},
                                                        tooltip={'placement': 'bottom', 'always_visible': True}),
                                        width=5),
                                dbc.Col(html.Label('HBA'), width=1),
                            ],
                            className="mb-3"
                        ),
                        dbc.Row(
                            [
                                dbc.Col(html.Label('Rings'), width=1),
                                dbc.Col(dcc.RangeSlider(id='Rings-slider', min=0, max=20, step=1, value=[0, 20],
                                                        marks={0: '0', 20: '20'},
                                                        tooltip={'placement': 'bottom', 'always_visible': True}),
                                        width=5),
                                dbc.Col(dcc.RangeSlider(id='ArRings-slider', min=0, max=10, step=1, value=[0, 10],
                                                        marks={0: '0', 10: '10'},
                                                        tooltip={'placement': 'bottom', 'always_visible': True}),
                                        width=5),
                                dbc.Col(html.Label('Ar. Rings'), width=1),
                            ],
                            className="mb-3"
                        ),
                        dbc.Row(
                            [
                                dbc.Col(html.Label('Rot. Bonds'), width=1),
                                dbc.Col(dcc.RangeSlider(id='RotBonds-slider', min=0, max=20, step=1, value=[0, 20],
                                                        marks={0: '0', 20: '20'},
                                                        tooltip={'placement': 'bottom', 'always_visible': True}),
                                        width=5),
                                dbc.Col(dcc.RangeSlider(id='Chiral-slider', min=0, max=10, step=1, value=[0, 10],
                                                        marks={0: '0', 10: '10'},
                                                        tooltip={'placement': 'bottom', 'always_visible': True}),
                                        width=5),
                                dbc.Col(html.Label('Chiral'), width=1),
                            ],
                            className="mb-3"
                        ),
                        dbc.Row(
                            [
                                dbc.Col(html.Label("Save as"), width=3),
                                dbc.Col(dbc.Input(type="text", id="save-as-input", placeholder="Filename", 
                                                  value="enumerated_molecules.csv", style={'fontSize': '14px'}),
                                        width=9)
                            ],
                            className="mb-3"
                        ),
                        dbc.Row(
                            [
                                dbc.Col(dbc.Button("Update", id="update-button", color="primary", className="mr-2",
                                                   style={'width': '100%', 'text-align': 'center'}), width=3),
                                dbc.Col(dbc.Button("Enumerate", id="enumerate-button", color="warning", className="mr-2",
                                                   style={'width': '100%', 'text-align': 'center'}), width=3),
                                dbc.Col(dbc.Button("Save", id="save-button", color="success",
                                                   style={'width': '100%', 'text-align': 'center'}), width=3)
                            ],
                            className="mb-3",
                            justify='end',
                        ),
                    ],
                    md=6,
                    style={'height': '100%', 'justify_content': 'center', 'margin': '-10px 0px 0px 0px'}
                ),
                dbc.Col(
                    [
                        html.H3("Molecule", style={'text-align': 'center'}),
                        html.Div(id='molecule-svg', style={'height': '150px', 'text-align': 'center'}),
                        html.H3("Enumeration Results", style={'text-align': 'center', 'margin': '20px 0px 20px 0px'}),
                        html.Div(dcc.Slider(id='enumeration-results-slider', min=0, max=0, step=1, value=0,
                                            tooltip={'placement': 'bottom', 'always_visible': False}), 
                                 style={'width': '70%', 'justify-content': 'center', 'margin': '0 auto'}),
                        html.Div(id='enumeration-svg', style={'height': '150px', 'text-align': 'center'}),
                        dcc.Download(id="download-enumerations"),
                        dcc.Store(id='enumeration-store')
                    ],
                    md=6,
                    style={'height': '100%', 'justify_content': 'center', 'margin': '20px 0px 0px 0px'}
                ),
            ],
            style={'height': '100%', 'justify_content': 'center', 'margin': '40px 0px 0px 0px'}
        ),
    ],
    fluid=True,
    style={'width': '80%'}
)

@app.callback(
    [Output('molecule-svg', 'children'),
     Output('molecule-svg', 'style'),
     Output('enumeration-svg', 'children'),
     Output('enumeration-svg', 'style'),
     Output('enumeration-results-slider', 'max'),
     Output('enumeration-results-slider', 'marks'),
     Output('enumeration-store', 'data'),
     Output('download-enumerations', 'data')],
    [Input('update-button', 'n_clicks'),
     Input('enumerate-button', 'n_clicks'),
     Input('enumeration-results-slider', 'value'),
     Input('save-button', 'n_clicks')],
    [State('molecule-input', 'value'),
     State('bb-source-select', 'value'),
     State('reaction-sites-input', 'value'),
     State('reaction-tags-dropdown', 'value'),
     State('struct-rules-input', 'value'),
     State('MW-slider', 'value'),
     State('TPSA-slider', 'value'),
     State('HBD-slider', 'value'),
     State('HBA-slider', 'value'),
     State('Rings-slider', 'value'),
     State('ArRings-slider', 'value'),
     State('RotBonds-slider', 'value'),
     State('Chiral-slider', 'value'),
     State('save-as-input', 'value'),
     State('enumeration-store', 'data')]
)
def update_output(update_clicks, enumerate_clicks, enumeration_value, save_clicks, 
                  molecule, building_blocks, reaction_sites, reaction_tags, struct_rules, 
                  mw, tpsa, hbd, hba, rings, arrings, rotbonds, chiral, save_as, enumerated_molecules):
    ctx = callback_context
    if not ctx.triggered:
        mol_img = 'Click Update to View the Molecule'
        mol_style = {'width': '75%', 'height': '175px', 'text-align': 'center', 
                     'fontSize': '20px', 'fontFamily': 'sans-serif', 'alignItems': 'center', 
                     'justifyContent': 'center', 'display': 'flex', 'margin': '0 auto'}
        enum_img = 'Click Enumerate to View the Enumerations'
        enum_style = {'width': '75%', 'height': '200px', 'text-align': 'center', 
                     'fontSize': '20px', 'fontFamily': 'sans-serif', 'alignItems': 'center', 
                     'justifyContent': 'center', 'display': 'flex', 'margin': '0 auto'}
        return mol_img, mol_style, enum_img, enum_style, 0, {}, None, None
    else:
        button_id = ctx.triggered[0]['prop_id'].split('.')[0]
        if button_id == 'update-button':
            mol_img = html.Img(src=utils.get_svg_mol(molecule, show_idx=True),
                               style={'height': '100%', 'width': '80%', 'margin': '0 auto'})
            mol_style = {'height': '175px', 'text-align': 'center'}
            return mol_img, mol_style, no_update, no_update, no_update, no_update, no_update, no_update
        elif button_id == 'enumerate-button':
            sites = [int(i) for i in reaction_sites.replace(' ', '').split(',')] if reaction_sites else []
            substruct_rules = [s for s in struct_rules.replace(' ', '').split(',')] if struct_rules else []
            rules = {'MW': mw, 'TPSA': tpsa, 'HBD': hbd, 'HBA': hba, 'Rings': rings, 'ArRings': arrings,
                     'RotB': rotbonds, 'Chiral': chiral}
            enumerator = utils.custom_enumerator(molecule, building_blocks, sites, reaction_tags, rules, substruct_rules)
            if not enumerator.enumerated_molecules:
                mol_img = html.Img(src=utils.get_svg_mol(molecule, show_idx=True),
                                   style={'height': '100%', 'width': '80%', 'margin': '0 auto'})
                mol_style = {'height': '175px', 'text-align': 'center'}
                slider_max = 0
                slider_marks = {0: '0'}
                enum_img = "No Enumerations! Check Inputs!"
                enum_style = {'backgroundColor': 'firebrick', 'width': '75%', 'height': '150px', 
                              'text-align': 'center', 'fontSize': '24px', 'fontFamily': 'sans-serif', 
                              'borderRadius': '20px', 'alignItems': 'center', 'justifyContent': 'center', 
                              'display': 'flex', 'margin': '0 auto'}
                return mol_img, mol_style, enum_img, enum_style, slider_max, slider_marks, [], no_update
            else:
                mol_img = html.Img(src=utils.get_svg_mol(molecule, show_idx=True),
                                   style={'height': '100%', 'width': '75%', 'margin': '0 auto'})
                mol_style = {'height': '175px', 'text-align': 'center'}
                enum_img = "Slide to view enumerated molecules!"
                enum_style = {'backgroundColor': 'forestgreen', 'width': '80%', 'height': '150px', 
                              'text-align': 'center', 'fontSize': '24px', 'fontFamily': 'sans-serif', 
                              'borderRadius': '20px', 'alignItems': 'center', 'justifyContent': 'center', 
                              'display': 'flex', 'margin': '0 auto'}
                slider_max = len(enumerator.enumerated_molecules) - 1
                n = len(enumerator.enumerated_molecules)
                slider_marks = {i: str(i) for i in range(0, n, max(10, round(n//10, -1)))}
                enum_mols = enumerator.enumerated_molecules
                return mol_img, mol_style, enum_img, enum_style, slider_max, slider_marks, enum_mols, no_update
        elif button_id == 'enumeration-results-slider':
            if enumerated_molecules:
                img = utils.get_svg_mol_with_bbs(*enumerated_molecules[enumeration_value][:2],
                                                 molecule,
                                                 bb_colors=['purple', 'green'],
                                                 legend=enumerated_molecules[enumeration_value][2])
                enum_img = html.Img(src=img,
                                    style={'height': '100%', 'width': '80%', 'margin': '0 auto'})
                enum_style = {'height': '200px', 'text-align': 'center'}
            else:
                enum_img = "No Enumerations! Couldn't Save!"
                enum_style = {'backgroundColor': 'firebrick', 'width': '75%', 'height': '150px', 
                              'text-align': 'center', 'fontSize': '24px', 'fontFamily': 'sans-serif', 
                              'borderRadius': '20px', 'alignItems': 'center', 'justifyContent': 'center', 
                              'display': 'flex', 'margin': '0 auto'}
            return no_update, no_update, enum_img, enum_style, no_update, no_update, no_update, no_update
        elif button_id == 'save-button':
            enum_img = f'Saved to {save_as}!'
            enum_style = {'backgroundColor': 'forestgreen', 'width': '75%', 'height': '150px',
                          'text-align': 'center', 'fontSize': '24px', 'fontFamily': 'sans-serif',
                          'borderRadius': '20px', 'alignItems': 'center', 'justifyContent': 'center', 
                          'display': 'flex', 'margin': '0 auto'}
            if enumerated_molecules:
                save_out = dcc.send_data_frame(utils.custom_enumerator_download_df(enumerated_molecules).to_csv, filename=save_as)
            else:
                save_out = None
            return no_update, no_update, enum_img, enum_style, no_update, no_update, no_update, save_out

if __name__ == '__main__':
    app.run(debug=True, jupyter_mode='tab')
